In [11]:
# 该notebook主要分为数据清洗或数据预处理(分别结合CDA2级教材第8章8.5逻辑回归内容)
# 导入必备库(结合CDA2级教程相关框架及思路)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
#读取数据集,测试集和训练集导入(由于kaggle数据集已经处理并分类,此处导入即可)
train=pd.read_csv(r"C:\Users\范彬\OneDrive\桌面\train.csv")
test=pd.read_csv(r"C:\Users\范彬\OneDrive\桌面\test.csv")#此处为个人数据集存放地址
print('训练集样本量 : %i \n 测试集样本量 ： %i'  %(len(train),len(test)))#个人进行数据集(训练集,测试集)体量统计
#首先,结合CDA2级教材思路,个人先对amt(金额)变量进行一元论逻辑回归模型建模(使用训练集)
lg=smf.logit("is_fraud ~ amt",train).fit()
lg.summary()
#结合CDA2级教程思路及一元逻辑回归模型结果,做出如下分析结论:
# 该模型生成信息分为二个部分,第一部分为模型的基本信息,第二部分是模型的参数估计与检验。
# 1.amt(金额)的系数为0.0028,而且p值显示系数不显著,但由于amt(金额)波动幅度相对较大,不能说明amt(金额)对欺诈影响不明显,仅显示影响幅度一般
# 2.结合CDA2级教程思路,计算OR值得e**0.0028=1.0028,即说明发生比的比值大于1，amt(金额每参加1个单位后欺诈率的可能性是原来的1.0028倍)
#结合CDA2级教程思路,多元逻辑回归中变量筛选阶段方法分为:向前回归法,向后回归法以及逐步回归法,此处借鉴书上使用的AIC准则进行的向前回归法变量筛选
# 自定义函数(复现CDA二级第八章8.5逻辑回归P219页)如下:
def forward_select(data,response):
    remaining=set(data.columns)
    remaining.remove(response)
    selected=[]
    current_score,beat_new_score=float("inf"),float("inf")
    while remaining:
        aic_with_candidates=[]
        for candidate in remaining:
            formula="{}~{}".format(response,"+".join(selected+[candidate]))
            aic=smf.logit(formula=formula,data=data).fit().aic
            aic_with_candidates.append((aic,candidate))
        aic_with_candidates.sort(reverse=True)
        best_new_score,best_candidate=aic_with_candidates.pop()
        if current_score>beat_new_score:
            remaining.remove(best_candidate)
            selected.append(best_candidate)
            current_score=best_new_score
            print("arc is {},continuing!".format(current_score))
        else:
            print("forward selection over!")
            break
    formula="{}~{}".format(response,"+".join(selected))
    print("final formula is {}".format(formula))
    model=smf.logit(formula=formula,data=data).fit()
    return model
# 以上为对CDA2级教程自定义函数的复现,但并不适用于本大体量数据集(1000000行+),以下仅供学生思路分享,大一学生学习能力有限,可能包含差错。
#接下来应用CDA2级书上使用的自定义函数即向前回归法进行对连续变量的筛选
# train_sample=train.sample(n=1000,random_state=42)#个人学生在第一次使用原数据集协方差矩阵计算,计算量过大,尝试使用取样,但出现取样is_fraud变量全部为空值，导致报错现象,故标注释化体现学生个人在学习使用CDA2级时第8章8.5.2逻辑回归模型及实现时复现线性回归法出现报错,主要原因为使用数据集体量过大及该回归模型为协方差矩阵，计算量极大,而且使用抽样法时response变量is_fraud为二分类变量,随机取样容易取空导致报错，在日后优化及版本更新过程中将使用sklearn等优化方法
#只有连续变量可以通过逐步回归进行筛选
# candidates=["amt","lat","long","merch_lat","merch_long","city_pop","trans_num","unix_time"]#沿用该数据集中的连续变量
# data_for_select=train_sample[candidates]
# lg_m1=forward_select(data=data_for_select,response="is_fraud")
# lg_m1.summary()
# print(f"原来有{len(candidates)-1}个变量")
# print(f"筛选剩下{len(lg_m1.index.tolist())}个(包含intercept 截距项)。")
# 由于数据集体量过大,使用抽样方法会导致抽取样本is_fraud类全部为0,导致直接报错

#接下来结合CDA2级教程思路,对分类变量进行显著性分析
# 补充CDA2级相关知识点，针对分类变量3种主要解决方案为:
# 1.逐一进行变量的显著性测试。
#2.使用sklearn中的决策树等方法筛选变量
# 3.使用WoE转换,通常需要进行恰当的归并或分箱。

# 接下来通过显著性测试逐一判断分类变量的显著性
class_col=["merchant","category","first","last","gender","street","city","state","zip","job"]
for i in class_col:
    tab=pd.crosstab(train[i],train["is_fraud"])
    print(i,""" p-value = %6.4f"""  %stats.chi2_contingency(tab)[1])
#得到分类变量p值均为0,属于显著，由于样本体量过大(1000000+行),本学生尝试使用抽样法进行尝试
train_sample=train.sample(n=1000,random_state=42)
for i in class_col:
    tab1=pd.crosstab(train_sample[i],train_sample["is_fraud"])
    print(i,""" p-value = %6.4f"""  %stats.chi2_contingency(tab)[1])
# 得到相同结果,在日后优化及版本更新过程中结合ai算法推荐及资料查询将使用IV值或者分箱法等优化方法

训练集样本量 : 1296675 
 测试集样本量 ： 555719
Optimization terminated successfully.
         Current function value: 0.032060
         Iterations 9
merchant  p-value = 0.0000
category  p-value = 0.0000
first  p-value = 0.0000
last  p-value = 0.0000
gender  p-value = 0.0000
street  p-value = 0.0000
city  p-value = 0.0000
state  p-value = 0.0000
zip  p-value = 0.0000
job  p-value = 0.0000
merchant  p-value = 0.0000
category  p-value = 0.0000
first  p-value = 0.0000
last  p-value = 0.0000
gender  p-value = 0.0000
street  p-value = 0.0000
city  p-value = 0.0000
state  p-value = 0.0000
zip  p-value = 0.0000
job  p-value = 0.0000
